# Tutorial Pipeline Pengenalan CAPTCHA Deep Learning (CRNN-CTC)
### Alur Kerja Berkas Tunggal: Pemrosesan Citra Masukan (`default.png`) Menuju Arsip (`solved/`)

---

## 1. Pendahuluan Alur Otomasi Pemrosesan Citra
Notebook ini meniru **alur otomasi pengenalan pola citra verifikasi berbasis teks**:
1. Sistem crawler mengunduh satu citra CAPTCHA baru dan menyimpannya sebagai **`default.png`** (atau berkas target masukan lainnya).
2. Citra tersebut diproses melalui serangkaian tahapan pra-pemrosesan citra digital (Alpha Blending, Bilinear Resizing, Grayscale, Min-Max Scaling, dan Penataan Tensor NCHW).
3. Model Deep Learning (CRNN) mengeksekusi inferensi untuk membaca urutan karakter dan menghitung **persentase keakuratan (*confidence score*)**.
4. Teks hasil pembacaan direkonstruksi menggunakan algoritma **CTC Greedy Decoding**.
5. Berkas fisik asli dipindahkan dan dinamai ulang secara otomatis ke folder **`solved/{teks_hasil}.png`**.

```
[Berkas Masukan: default.png]
              │
              ▼
[Tahap 1: Alpha Channel Blending] ────────> Netralkan transparansi PNG menjadi putih solid (255)
              │
              ▼
[Tahap 2: Bilinear Resizing] ─────────────> Standarisasi tinggi H=64 px, skala lebar W proporsional
              │
              ▼
[Tahap 3: Grayscale Conversion] ──────────> Reduksi ruang warna 3-kanal ke 1-kanal keabuan (L-channel)
              │
              ▼
[Tahap 4: Min-Max Normalization] ─────────> Skalakan matriks piksel [0, 255] -> float32 [0.0, 1.0]
              │
              ▼
[Tahap 5: NCHW Tensor Reshaping] ─────────> Susun dimensi memori tensor 4D [1, 1, 64, W]
              │
              ▼
[Tahap 6: Inferensi CRNN pada ONNX] ──────> Forward pass lapisan konvolusi + BiLSTM pada CPU
              │
              ▼
[Tahap 7: Softmax & Keakuratan] ──────────> Hitung sebaran probabilitas & persentase keakuratan (%)
              │
              ▼
[Tahap 8: CTC Greedy Decoding] ───────────> Runtuhkan duplikasi token berurutan & buang blank token
              │
              ▼
[Tahap 9: Eksekusi Pemindahan Berkas] ────> Pindahkan berkas fisik ke folder 'solved/{teks}.png'
```


---
### Persiapan Dependensi Kernel Jupyter
Jika Anda menjalankan notebook ini untuk pertama kali atau pada kernel baru, jalankan perintah instalasi berikut:


In [ ]:
# Instalasi dependensi langsung ke kernel aktif Jupyter:
%pip install numpy Pillow onnxruntime matplotlib


In [ ]:
# 1. Pemuatan & Verifikasi Otomatis Dependensi Pustaka
import sys
import subprocess

# Auto-check & install dependensi jika kernel Jupyter belum memilikinya
required_packages = {
    'numpy': 'numpy',
    'PIL': 'Pillow',
    'onnxruntime': 'onnxruntime',
    'matplotlib': 'matplotlib'
}
for mod_name, pip_name in required_packages.items():
    try:
        __import__(mod_name)
    except ImportError:
        print(f"[!] Modul '{mod_name}' belum terpasang di kernel ini ({sys.executable}).")
        print(f"[*] Memasang otomatis '{pip_name}'...")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pip_name])

# Import pustaka standar dan visi komputer
import os
import shutil
import base64
import zlib
from typing import List, Tuple, Union, Optional

import numpy as np
from PIL import Image
import onnxruntime as ort
import matplotlib.pyplot as plt

# Konfigurasi visualisasi matplotlib
plt.rcParams['figure.figsize'] = (10, 3)
plt.rcParams['image.cmap'] = 'gray'

print("=" * 60)
print("Seluruh pustaka berhasil diverifikasi dan dimuat!")
print(f"Interpreter Kernel : {sys.executable}")
print(f"NumPy Version      : {np.__version__}")
print(f"Pillow Version     : {Image.__version__}")
print(f"ONNX Runtime       : {ort.__version__}")
print(f"Matplotlib Version : {plt.matplotlib.__version__}")
print("=" * 60)


---
## Langkah 0: Pemuatan Model ONNX & Kamus Karakter (Charset) Mandiri

### Alasan Digunakan:
Model neural network `model.onnx` berisi representasi bobot teroptimasi dari arsitektur CRNN (CNN + BiLSTM). Bobot ini dimuat langsung menggunakan mesin inferensi ONNX Runtime yang sangat efisien pada CPU.

### Fungsinya untuk Apa?
1. **`model.onnx`**: Menyediakan fungsi pemetaan fitur visual ke probabilitas karakter.
2. **`_CHARSET_DATA_B64`**: Kamus internal berisi 8.210 entri karakter yang didekompresi langsung ke memori (tanpa membaca file eksternal), di mana indeks 0 adalah *Blank Token* ($\epsilon$).


In [ ]:
# Kamus karakter terkompresi internal (8.210 entri)
_CHARSET_DATA_B64 = "..."

# Mendekompresi charset langsung dari memori
raw_bytes = base64.b64decode(_CHARSET_DATA_B64.encode('ascii'))
CHARSET = zlib.decompress(raw_bytes).decode('utf-8').split('\n')

print(f"[*] Total Karakter dalam Charset : {len(CHARSET)} entri")
print(f"[*] Karakter pada Indeks 0      : '{CHARSET[0]}' (Blank Token / Epsilon)")

# Path berkas model ONNX lokal
MODEL_PATH = os.path.abspath("model.onnx")
if not os.path.exists(MODEL_PATH):
    MODEL_PATH = os.path.abspath("models/model.onnx")

print(f"[*] Jalur Berkas Model ONNX     : {MODEL_PATH}")

# Inisialisasi sesi inferensi ONNX Runtime
session_opts = ort.SessionOptions()
session_opts.log_severity_level = 3  # Menekan log internal level rendah
ort_session = ort.InferenceSession(MODEL_PATH, sess_options=session_opts, providers=['CPUExecutionProvider'])

input_name = ort_session.get_inputs()[0].name
input_shape = ort_session.get_inputs()[0].shape
output_name = ort_session.get_outputs()[0].name
output_shape = ort_session.get_outputs()[0].shape

print(f"[*] Model Input Name & Shape     : {input_name} -> {input_shape}")
print(f"[*] Model Output Name & Shape    : {output_name} -> {output_shape}")


---
## Langkah 1: Penentuan Berkas Target Masukan (`target_file`) & Validasi Ketersediaan Berkas

### Alasan Digunakan:
Dalam alur kerja pemrosesan CAPTCHA, sistem membaca **satu berkas target masukan** (secara *default* bernama **`default.png`**).
Ketersediaan fisik berkas ini harus divalidasi terlebih dahulu sebelum sistem menjalankan tahapan pra-pemrosesan citra digital.

### Fungsinya untuk Apa?
1. Menentukan nama/jalur berkas yang akan diolah (`target_file`).
2. **Pemeriksaan Fisik Ketersediaan Berkas**: Jika berkas tidak ditemukan pada direktori kerja, sistem langsung **menampilkan pesan bahwa file yang dimaksud tidak ada**, tanpa memicu *error crash* tak terduga.
3. Jika berkas tersedia, memuat citra via Pillow dan menampilkan visualisasi resolusi asli $(W \times H)$, format berkas, serta mode saluran warna.


In [ ]:
# Tentukan berkas masukan yang akan diproses (default: 'default.png')
target_file = "default.png"

# Penyesuaian jalur jika notebook dijalankan dari root workspace
if not os.path.exists(target_file) and os.path.exists(os.path.join("captcha", "bsi", target_file)):
    target_file = os.path.join("captcha", "bsi", target_file)

img_original = None

# Pemeriksaan fisik berkas: Jika tidak ada, tampilkan pesan bahwa file yang dimaksud tidak ada
if not os.path.exists(target_file):
    print("=" * 65)
    print(f"[!] PEMBERITAHUAN: File yang dimaksud tidak ada!")
    print("=" * 65)
    print(f"Target Berkas   : '{target_file}'")
    print(f"Direktori Aktif : {os.path.abspath(os.getcwd())}")
    print(f"Status          : Berkas tidak ditemukan pada direktori kerja.")
    print("Silakan letakkan berkas yang dimaksud terlebih dahulu untuk melanjutkan.")
    print("=" * 65)
else:
    # Memuat dan menginspeksi citra jika berkas ada
    img_original = Image.open(target_file)
    print(f"[*] Berkas Target Aktif : {target_file}")
    print(f"[*] Resolusi Asli       : {img_original.size[0]} x {img_original.size[1]} piksel (Lebar x Tinggi)")
    print(f"[*] Format Berkas       : {img_original.format}")
    print(f"[*] Mode Saluran Warna  : {img_original.mode}")

    # Visualisasi citra target masukan
    plt.figure(figsize=(4, 2))
    plt.imshow(img_original)
    plt.title(f"Citra Masukan Target: {os.path.basename(target_file)} ({img_original.mode})")
    plt.axis('off')
    plt.show()


---
## Langkah 2: Normalisasi Transparansi (Alpha Channel Blending)

### Alasan Digunakan:
Citra format PNG sering kali menyimpan area latar belakang sebagai transparan (kanal alfa $A = 0$). Pada data mentah piksel, warna di area transparan tersebut sering kali bernilai $(0, 0, 0)$ (hitam). Jika kanal alfa diabaikan begitu saja saat konversi, latar belakang citra akan berubah menjadi **hitam pekat** sehingga teks tertutup.

### Fungsinya untuk Apa?
Mendeteksi setiap koordinat piksel dengan $A(x, y) = 0$ dan mengganti warnanya menjadi putih solid $(255, 255, 255)$:

$$\mathbf{I}_{\text{out}}(x, y) = \begin{cases} (255, 255, 255) & \text{jika } A(x, y) = 0 \\ \mathbf{I}_{\text{in}}(x, y)_{[:3]} & \text{jika } A(x, y) > 0 \end{cases}$$


In [ ]:
def apply_alpha_blending(img: Image.Image) -> Image.Image:
    """
    Mengganti area transparan (Alpha channel) dengan latar belakang putih solid (255, 255, 255).
    """
    if img.mode in ("RGBA", "LA") or (img.mode == "P" and "transparency" in img.info):
        alpha = img.convert("RGBA").split()[-1]
        bg = Image.new("RGBA", img.size, (255, 255, 255, 255))
        bg.paste(img, mask=alpha)
        return bg.convert("RGB")
    return img.convert("RGB")

if 'img_original' not in globals() or img_original is None:
    print(f"[!] Langkah 2 dilewati karena file '{target_file}' tidak ada.")
    img_blended = None
else:
    # Eksekusi Tahap 1: Alpha channel blending
    img_blended = apply_alpha_blending(img_original)

    # Perbandingan Visual Sebelum vs Sesudah Alpha Blending
    fig, axes = plt.subplots(1, 2, figsize=(8, 2.5))
    axes[0].imshow(img_original)
    axes[0].set_title(f"Sebelum: {img_original.mode} (Bisa Transparan/Gelap)")
    axes[0].axis('off')

    axes[1].imshow(img_blended)
    axes[1].set_title("Sesudah: RGB Solid (Latar Putih Bersih)")
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()


---
## Langkah 3: Penskalaan Spasial Aspek-Rasio Terjaga (Bilinear Resizing H=64)

### Alasan Digunakan:
Jaringan konvolusional CRNN mensyaratkan dimensi tinggi citra tetap **$H = 64$ piksel** agar cocok dengan ukuran kernel filter vertikal. Namun, lebar citra bervariasi sesuai panjang karakter, sehingga rasio aspek asli harus dipertahankan agar huruf tidak terkompresi atau terdistorsi secara fisik.

### Fungsinya untuk Apa?
1. Mengunci tinggi citra tepat $64$ piksel.
2. Menghitung lebar target secara proporsional:
   $$W_{\text{target}} = \text{int}\left(W_{\text{asli}} \times \frac{64}{H_{\text{asli}}}\right)$$
3. Melakukan interpolasi bilinier (*Bilinear Interpolation*) untuk meratakan gradien warna pada tepi karakter.


In [ ]:
def resize_keep_aspect_ratio(img: Image.Image, target_h: int = 64) -> Image.Image:
    """
    Melakukan penskalaan citra dengan mengunci tinggi tepat pada target_h (64 px)
    dan menghitung lebar proporsional menggunakan interpolasi bilinier.
    """
    w, h = img.size
    scale = target_h / float(h)
    new_w = max(1, int(round(w * scale)))
    return img.resize((new_w, target_h), Image.Resampling.BILINEAR)

if 'img_blended' not in globals() or img_blended is None:
    print(f"[!] Langkah 3 dilewati karena file '{target_file}' tidak ada.")
    img_resized = None
else:
    # Eksekusi Tahap 2: Resizing bilinier H=64
    img_resized = resize_keep_aspect_ratio(img_blended, target_h=64)

    print(f"[*] Dimensi Sebelum Resizing : {img_blended.size[0]} x {img_blended.size[1]} px")
    print(f"[*] Dimensi Sesudah Resizing : {img_resized.size[0]} x {img_resized.size[1]} px (Tinggi tepat 64 px)")

    plt.figure(figsize=(5, 2))
    plt.imshow(img_resized)
    plt.title(f"Hasil Penskalaan Bilinier H=64: {img_resized.size[0]}x{img_resized.size[1]} px")
    plt.axis('off')
    plt.show()


---
## Langkah 4: Reduksi Ruang Warna Menuju Skala Keabuan (Grayscale L-Channel)

### Alasan Digunakan:
Pengenalan huruf dan angka mengandalkan fitur tepi morfologis, bukan warna. Tiga kanal RGB hanya membebani memori tanpa menambah informasi diskriminatif.

### Fungsinya untuk Apa?
Mengonversi citra menjadi 1-kanal skala keabuan (*Luminance*) berstandar ITU-R BT.601:

$$Y(x, y) = 0.299 \cdot R(x, y) + 0.587 \cdot G(x, y) + 0.114 \cdot B(x, y)$$

Menghemat beban memori dan komputasi hingga **66.7%**.


In [ ]:
if 'img_resized' not in globals() or img_resized is None:
    print(f"[!] Langkah 4 dilewati karena file '{target_file}' tidak ada.")
    img_gray = None
    arr_gray = None
else:
    # Eksekusi Tahap 3: Konversi ke skala keabuan 1-kanal 'L'
    img_gray = img_resized.convert("L")
    arr_rgb = np.array(img_resized)
    arr_gray = np.array(img_gray)

    print(f"[*] Matriks RGB 3-Kanal       : {arr_rgb.shape}")
    print(f"[*] Matriks Grayscale 1-Kanal : {arr_gray.shape}")
    print(f"[*] Penghematan Memori        : {(1 - arr_gray.nbytes / arr_rgb.nbytes) * 100:.1f}%")

    plt.figure(figsize=(4, 2))
    plt.imshow(arr_gray, cmap='gray')
    plt.title("Citra 1-Kanal Skala Keabuan (Grayscale 'L')")
    plt.axis('off')
    plt.show()


---
## Langkah 5: Normalisasi Intensitas Numerik (Min-Max Scaling $[0.0, 1.0]$ Float32)

### Alasan Digunakan:
Nilai intensitas integer 8-bit $[0, 255]$ memicu ketidakstabilan gradien dan saturasi fungsi aktivasi jaringan saraf.

### Fungsinya untuk Apa?
Memetakan seluruh nilai piksel ke rentang `float32` $[0.0, 1.0]$ dengan membaginya skalar dengan $255.0$:

$$\mathbf{I}_{\text{norm}}(x, y) = \frac{\mathbf{I}(x, y)}{255.0}$$


In [ ]:
if 'arr_gray' not in globals() or arr_gray is None:
    print(f"[!] Langkah 5 dilewati karena file '{target_file}' tidak ada.")
    arr_normalized = None
else:
    # Eksekusi Tahap 4: Min-Max Intensity Scaling
    arr_normalized = arr_gray.astype(np.float32) / 255.0

    print(f"[*] Rentang Piksel Asli          : Min = {arr_gray.min()}, Max = {arr_gray.max()} (uint8)")
    print(f"[*] Rentang Piksel Ternormalisasi: Min = {arr_normalized.min():.4f}, Max = {arr_normalized.max():.4f} (float32)")

    # Visualisasi histogram distribusi nilai piksel
    plt.figure(figsize=(6, 2))
    plt.hist(arr_normalized.ravel(), bins=50, color='royalblue', alpha=0.75, edgecolor='black')
    plt.title("Distribusi Nilai Piksel Ternormalisasi [0.0, 1.0]")
    plt.xlabel("Intensitas Numerik (Float32)")
    plt.ylabel("Frekuensi Piksel")
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()


---
## Langkah 6: Penataan Topologi Dimensi Tensor 4D (NCHW)

### Alasan Digunakan:
ONNX Runtime mensyaratkan tensor input berdimensi 4D dalam urutan $(N, C, H, W)$:
- $N=1$ (Batch Size)
- $C=1$ (Saluran Grayscale)
- $H=64$ (Tinggi Spasial)
- $W$ (Lebar Proporsional)

### Fungsinya untuk Apa?
Mentransformasikan matriks 2D $(64, W)$ menjadi tensor 4D `[1, 1, 64, W]` menggunakan `np.expand_dims()`.


In [ ]:
if 'arr_normalized' not in globals() or arr_normalized is None:
    print(f"[!] Langkah 6 dilewati karena file '{target_file}' tidak ada.")
    tensor_nchw = None
else:
    # Eksekusi Tahap 5: Penambahan Dimensi Batch dan Channel
    tensor_nchw = np.expand_dims(np.expand_dims(arr_normalized, axis=0), axis=0)

    print(f"[*] Bentuk Matriks 2D Awal : {arr_normalized.shape}")
    print(f"[*] Bentuk Tensor 4D Akhir : {tensor_nchw.shape} (Format NCHW: [N=1, C=1, H=64, W={tensor_nchw.shape[3]}])")
    print(f"[*] Tipe Data Memori Tensor: {tensor_nchw.dtype} ({tensor_nchw.nbytes} bytes)")


---
## Langkah 7: Inferensi Komputasi Deep Learning pada ONNX Runtime (Forward Pass)

### Alasan Digunakan:
Mengekstrak peta fitur spasial menggunakan CNN dan menangkap ketergantungan urutan sekuensial waktu menggunakan BiLSTM.

### Fungsinya untuk Apa?
Menghasilkan matriks *logits* mentah tak terikat yang merepresentasikan skor aktivasi untuk setiap karakter pada setiap langkah waktu (*time-step*).


In [ ]:
if 'tensor_nchw' not in globals() or tensor_nchw is None:
    print(f"[!] Langkah 7 dilewati karena file '{target_file}' tidak ada.")
    raw_logits = None
    logits_2d = None
else:
    # Eksekusi Inferensi Forward Pass pada Model ONNX
    raw_outputs = ort_session.run(None, {input_name: tensor_nchw})
    raw_logits = raw_outputs[0]

    print(f"[*] Dimensi Output Mentah Jaringan : {raw_logits.shape}")
    seq_len = raw_logits.shape[0]
    num_classes = raw_logits.shape[2]
    logits_2d = raw_logits[:, 0, :]

    print(f"[*] Panjang Irisan Waktu (Time-steps / Frames) : {seq_len}")
    print(f"[*] Total Kelas Karakter (Vocabulary Classes)   : {num_classes}")


---
## Langkah 8: Fungsi Aktivasi Softmax & Perhitungan Persentase Keakuratan

### Alasan Digunakan:
Skor logits bernilai riil $(-\infty, +\infty)$ sehingga belum mencerminkan tingkat kepastian model. Kita perlu memproyeksikannya menjadi probabilitas $[0.0, 1.0]$.

### Fungsinya untuk Apa?
1. Menerapkan Softmax terstabilkan secara numerik:
   $$P(c_t \mid \mathbf{x}) = \frac{e^{z_{t, c} - \max(z_t)}}{\sum_{j} e^{z_{t, j} - \max(z_t)}}$$
2. Mengambil probabilitas tertinggi (*max probability*) pada setiap langkah waktu.
3. Menghitung rata-rata probabilitas dari karakter yang berhasil dibaca sebagai **persentase tingkat keakuratan (*confidence score %*)**.


In [ ]:
if 'logits_2d' not in globals() or logits_2d is None:
    print(f"[!] Langkah 8 dilewati karena file '{target_file}' tidak ada.")
    best_indices = None
    max_probabilities = None
else:
    # Softmax terstabilkan secara numerik
    exp_logits = np.exp(logits_2d - np.max(logits_2d, axis=-1, keepdims=True))
    probabilities = exp_logits / np.sum(exp_logits, axis=-1, keepdims=True)

    # Ambil indeks kelas terbaik dan probabilitasnya di setiap langkah waktu
    best_indices = np.argmax(probabilities, axis=-1)
    max_probabilities = np.max(probabilities, axis=-1)

    print("Tabel Probabilitas Tertinggi pada 10 Langkah Waktu Pertama:")
    print("-" * 55)
    print(f"{'Step':<6} | {'Indeks':<8} | {'Probabilitas':<14} | {'Karakter'}")
    print("-" * 55)
    for t in range(min(10, seq_len)):
        idx = best_indices[t]
        char = repr(CHARSET[idx]) if idx < len(CHARSET) else 'N/A'
        print(f"{t:<6} | {idx:<8} | {max_probabilities[t]:.6f}       | {char}")


---
## Langkah 9: CTC Greedy Decoding (Rekonstruksi Teks Sekuensial)

### Alasan Digunakan:
Sebuah huruf fisik pada citra membentang di beberapa langkah waktu horizontal, sehingga neural network mengeluarkan prediksi huruf yang berulang pada frame berturutan.

### Fungsinya untuk Apa?
1. Meruntuhkan duplikasi karakter identik yang berdampingan (`'m', 'm', 'm'` $\rightarrow$ `'m'`).
2. Membuang *blank token* ($\epsilon$, indeks 0) yang memisahkan karakter kembar.
3. Menghitung rata-rata skor probabilitas karakter yang lolos seleksi sebagai persentase keakuratan akhir:

$$\text{Keakuratan (\%)} = \left(\frac{1}{L} \sum_{i=1}^L P(c_i)\right) \times 100\%$$


In [ ]:
if 'best_indices' not in globals() or best_indices is None:
    print(f"[!] Langkah 9 dilewati karena file '{target_file}' tidak ada.")
    final_detected_text = ""
    confidence_percentage = 0.0
else:
    print("Tracing Eksekusi CTC Greedy Decoding pada Citra Target:")
    print("=" * 65)
    print(f"{'Step':<6} | {'Index':<8} | {'Prob':<10} | {'Aksi CTC':<20} | {'Teks Sementara'}")
    print("=" * 65)

    decoded_chars = []
    char_confidences = []
    prev_idx = None

    for t, (idx, prob) in enumerate(zip(best_indices, max_probabilities)):
        idx = int(idx)
        action = ""
        if idx == prev_idx:
            action = "Abaikan (Duplikat)"
        elif idx == 0:
            action = "Abaikan (Blank Token)"
        else:
            char = CHARSET[idx]
            decoded_chars.append(char)
            char_confidences.append(float(prob))
            action = f"TERIMA: '{char}' ({prob*100:.2f}%)"
        
        current_text = "".join(decoded_chars)
        if action != "Abaikan (Duplikat)":
            print(f"{t:<6} | {idx:<8} | {prob:.4f}     | {action:<20} | {current_text}")
        prev_idx = idx

    final_detected_text = "".join(decoded_chars).strip().lower()
    confidence_percentage = float(np.mean(char_confidences)) * 100.0 if char_confidences else 0.0

    print("=" * 65)
    print(f"HASIL PARSER AKHIR  : {final_detected_text}")
    print(f"TINGKAT KEAKURATAN  : {confidence_percentage:.2f}%")
    print("=" * 65)


---
## Langkah 10: Pemindahan dan Penamaan Ulang (Rename) ke Folder `solved/`

### Alasan Digunakan:
Setelah citra masukan berhasil diuraikan oleh model neural network dan teks hasil pembacaan (`final_detected_text`) diperoleh, berkas citra tersebut **diarsipkan ke folder `solved/` dengan nama baru sesuai teks hasil deteksi** (misalnya `solved/32617.png`).

### Fungsinya untuk Apa?
1. Memastikan direktori arsip tujuan (`solved/`) telah tersedia (`os.makedirs`).
2. Menyalin berkas citra ke folder `solved/` dengan nama baru (`shutil.copy2`). Berkas baru tersimpan di `solved/`, sedangkan berkas lama di direktori kerja siap dihapus pada langkah selanjutnya.


In [ ]:
if target_file is None or not os.path.exists(target_file) or not final_detected_text:
    print(f"[!] Langkah 10 dilewati: File masukan '{target_file}' tidak ada / belum dipecahkan.")
    destination_path = ""
else:
    # 1. Tentukan direktori arsip 'solved'
    script_dir = os.path.dirname(os.path.abspath(target_file))
    solved_directory = os.path.join(script_dir, "solved")
    os.makedirs(solved_directory, exist_ok=True)

    # 2. Dapatkan ekstensi berkas (.png, .jpg, dll)
    _, ext = os.path.splitext(target_file)
    if not ext:
        ext = ".png"

    # 3. Susun penamaan berkas baru berdasarkan teks hasil OCR
    new_filename = f"{final_detected_text}{ext}"
    destination_path = os.path.join(solved_directory, new_filename)

    # 4. Salin berkas target ke folder arsip dengan nama baru
    shutil.copy2(target_file, destination_path)

    print("=" * 65)
    print("STATUS LANGKAH 10: PEMINDAHAN & RENAME BERKAS")
    print("=" * 65)
    print(f"[*] Berkas Target Asal     : {target_file}")
    print(f"[*] Folder Arsip Tujuan    : {solved_directory}")
    print(f"[*] Berkas Baru di Solved  : {destination_path}")
    print(f"[*] Status Berkas Baru     : Tersimpan ({os.path.exists(destination_path)})")
    print(f"[✓] Berkas berhasil dipindahkan ke solved dan direname menjadi '{new_filename}'.")
    print("=" * 65)


---
## Langkah 11: Menghapus Berkas Lama di Direktori Kerja (`os.remove`)

### Alasan Digunakan:
Karena berkas citra **sudah dipindah ke folder `solved/` dan sudah direname** pada Langkah 10, maka berkas asli lama di direktori kerja (`default.png`) **wajib dihapus**.

Alasan penghapusan berkas lama:
1. **Mencegah Siklus Ganda (Duplicate Processing)**: Mencegah worker crawler membaca kembali berkas CAPTCHA lama yang sudah kedaluwarsa.
2. **Kebersihan Direktori Kerja (Directory Hygiene)**: Menjaga direktori operasional tetap bersih dari tumpukan berkas mentah.
3. **Pemberhentian Siklus Hidup Berkas Lama**: Berkas telah berganti identitas menjadi arsip terpecahkan di `solved/`.

### Fungsinya untuk Apa?
1. Memvalidasi bahwa berkas baru di `solved/` telah eksis sebelum berkas asal dihapus.
2. Mengeksekusi penghapusan fisik berkas lama dengan `os.remove(target_file)`.
3. Memverifikasi dengan `os.path.exists(target_file)` bernilai `False` (terhapus bersih).


In [ ]:
if target_file is None or not os.path.exists(target_file):
    print(f"[!] Langkah 11 dilewati: File lama '{target_file}' tidak ada / sudah terhapus.")
else:
    print("=" * 65)
    print("STATUS LANGKAH 11: PENGHAPUSAN BERKAS LAMA DI DIREKTORI KERJA")
    print("=" * 65)
    print(f"[*] Target Berkas Lama      : {target_file}")
    print(f"[*] Status Sebelum Dihapus  : os.path.exists('{target_file}') -> {os.path.exists(target_file)}")

    # 1. Validasi keamanan: Pastikan berkas di solved sudah aman tersimpan
    if not destination_path or not os.path.exists(destination_path):
        raise RuntimeError(f"ABORT: Berkas baru di '{destination_path}' tidak ditemukan! Penghapusan dibatalkan.")

    # 2. Eksekusi penghapusan berkas fisik lama menggunakan os.remove()
    os.remove(target_file)
    print(f"[✓] Perintah os.remove('{target_file}') berhasil dieksekusi.")

    # 3. Verifikasi fisik pasca-penghapusan
    is_old_still_exists = os.path.exists(target_file)
    status_str = "MASIH ADA (GAGAL HAPUS)" if is_old_still_exists else "TERHAPUS / BERSIH"
    print("-" * 65)
    print(f"[*] Status Akhir Berkas Lama: {status_str}")
    print(f"[*] Verifikasi Fisik        : os.path.exists('{target_file}') -> {is_old_still_exists}")
    if not is_old_still_exists:
        print(f"[✓] SUKSES: Berkas lama '{target_file}' telah dihapus dari direktori kerja.")
    print("=" * 65)


---
## Langkah 12: Fungsi Terpadu Produksi (`process_and_move_to_solved`)

### Alasan Digunakan:
Dalam aplikasi nyata (crawler bot atau microservice), seluruh alur dari Langkah 1 hingga Langkah 11 dienkapsulasi menjadi **satu fungsi modular yang siap pakai** sebagai *drop-in replacement*.

### Fungsinya untuk Apa?
1. Memeriksa ketersediaan berkas masukan (`image_path`). Jika file tidak ada, langsung menampilkan pesan bahwa file yang dimaksud tidak ada.
2. Mengeksekusi rangkaian normalisasi, inferensi ONNX, dan decoding CTC.
3. Memindahkan dan merename berkas ke `solved/{text}{ext}`, serta menghapus berkas lama dari direktori kerja secara bersih.
4. Mengembalikan tuple: `(teks_terdeteksi, jalur_berkas_baru, persentase_keakuratan)`.


In [ ]:
def process_and_move_to_solved(image_path: str, solved_dir: str) -> Tuple[str, str, float]:
    """
    Fungsi terpadu otomasi pipeline pengenalan citra CAPTCHA:
    1. Memvalidasi ketersediaan berkas masukan.
    2. Mengekstrak teks beserta persentase keakuratan dari citra masukan.
    3. Memastikan folder solved tersedia di disk.
    4. Memindahkan dan merename berkas ke solved/{teks}{ext}.
    5. Menghapus berkas lama dari direktori kerja secara bersih.
    
    Args:
        image_path: Jalur berkas gambar target (misal: default.png).
        solved_dir: Jalur folder arsip tujuan (misal: solved).
        
    Returns:
        Tuple[str, str, float]: (teks_hasil, jalur_berkas_baru, persentase_keakuratan)
    """
    # Pemeriksaan keberadaan berkas: jika tidak ada, tampilkan pesan
    if not os.path.exists(image_path):
        print(f"[!] Peringatan: File yang dimaksud tidak ada pada jalur '{image_path}'!")
        raise FileNotFoundError(f"File yang dimaksud tidak ada: '{image_path}'")
    
    # 1. Eksekusi pipeline normalisasi citra digital
    img = Image.open(image_path)
    img = apply_alpha_blending(img)
    img = resize_keep_aspect_ratio(img, target_h=64)
    img = img.convert("L")
    t_mat = np.array(img, dtype=np.float32) / 255.0
    t_in = np.expand_dims(np.expand_dims(t_mat, axis=0), axis=0)
    
    # 2. Forward pass komputasi inferensi ONNX
    raw = ort_session.run(None, {input_name: t_in})[0]
    l_2d = raw[:, 0, :] if raw.ndim == 3 else raw
    
    # 3. Softmax terstabilkan & CTC Greedy Decoding
    exp_l = np.exp(l_2d - np.max(l_2d, axis=-1, keepdims=True))
    probs = exp_l / np.sum(exp_l, axis=-1, keepdims=True)
    best_i = np.argmax(probs, axis=-1)
    max_p = np.max(probs, axis=-1)
    
    chars, confs, p_i = [], [], None
    for idx, p in zip(best_i, max_p):
        idx = int(idx)
        if idx != p_i and idx != 0 and idx < len(CHARSET):
            chars.append(CHARSET[idx])
            confs.append(float(p))
        p_i = idx
        
    text = "".join(chars).strip().lower()
    conf = float(np.mean(confs)) * 100.0 if confs else 0.0
    
    if not text:
        raise ValueError("Gagal mendeteksi teks atau hasil pengenalan karakter kosong.")
        
    # 4. Manajemen Folder Arsip, Pemindahan, dan Rename Berkas
    os.makedirs(solved_dir, exist_ok=True)
    _, ext = os.path.splitext(image_path)
    if not ext:
        ext = ".png"
    new_filename = f"{text}{ext}"
    dest_path = os.path.join(solved_dir, new_filename)
    
    # Salin berkas ke folder arsip baru
    shutil.copy2(image_path, dest_path)
    
    # Hapus berkas lama dari direktori kerja secara definitif
    if os.path.exists(image_path):
        os.remove(image_path)
        
    return text, dest_path, conf

print("[✓] Fungsi terpadu 'process_and_move_to_solved' berhasil dimuat dan siap digunakan secara modular.")
